In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics, preprocess_inference_results

results_dir = PROJECT_ROOT / "results"

## ICU Readmission results

In [1]:
def process_drg_prediction_results(input_dir, outcome_tokens, **kwargs) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual").is_in(outcome_tokens),
        expected_expr=pl.col("expected").is_in(outcome_tokens),
        filter_ambiguous=pl.col("stop_reason") == Reason.GOT_TOKEN,
        **kwargs,
    )

NameError: name 'pl' is not defined

In [1]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.mimic_icu import DrgPredictionDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = DrgPredictionDataset(path)

In [2]:
len(dataset)

46685

In [5]:
dataset[0][1]

{'expected': 'DRG//66',
 'hadm_id': 28439444,
 'true_token_dist': 1,
 'true_token_time': 0,
 'patient_id': 10002155,
 'data_idx': 351}

In [3]:
import json
hadm_ids = [d[1]['hadm_id'] for d in dataset]
with open("hadm_ids.json", "w") as f:
    json.dump(hadm_ids, f)

In [ ]:
drg_prediction_results = []
for input_dir in (results_dir / Task.DRG_PREDICTION).iterdir():
    df = process_drg_prediction_results(input_dir, outcome_tokens=[ST.ICU_ADMISSION])

    rep_num = df["counts"].mean()
    res = compute_and_print_metrics(
        df["expected"],
        df["actual"],
        f"ICU Readmission (rep_num={rep_num:.2f})\n{input_dir.name}",
    )
    drg_prediction_results.append(
        {
            "name": input_dir.name,
            "auc": res["fitted_auc"],
            "rep_num": rep_num,
        }
    )
    plt.show()
drg_prediction_results = pl.DataFrame(drg_prediction_results).sort("auc", descending=True)

In [ ]:
drg_prediction_results